# 02 — Wave Data Prototype (CMEMS)

**Story**: S01-02 | **Spec**: `openspec/capabilities/waves/spec.md`

Retrieves significant wave height (VHM0) and mean wave direction (VMDR) from the
Copernicus Marine Service (CMEMS) Baltic Sea Wave Analysis and Forecast product
for a Warnemünde AOI. Establishes auth and query patterns for the Phase 2 FastAPI service.

**Product**: `BALTICSEA_ANALYSISFORECAST_WAV_003_010`  
**AOI**: German Baltic coast — bbox `[9.5, 53.5, 15.0, 56.0]`  
**Auth**: Copernicus account via `COPERNICUSMARINE_SERVICE_USERNAME/PASSWORD` in `.env`  

> CMEMS and CDSE now use the same Copernicus account since the 2023 service merger.

In [ ]:
import os
from pathlib import Path
from datetime import datetime, timezone, timedelta
import numpy as np
import xarray as xr
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from dotenv import load_dotenv
import copernicusmarine as cm

# Load .env (for CDSE and other services)
load_dotenv(Path("../.env"))

# CMEMS credentials — checked from env vars OR cached credentials file
# (~/.copernicusmarine/.copernicusmarine-credentials, written by `copernicusmarine login`)
# If env vars are empty, copernicusmarine falls back to the cached file automatically.
USERNAME = os.getenv("COPERNICUSMARINE_SERVICE_USERNAME") or None
PASSWORD = os.getenv("COPERNICUSMARINE_SERVICE_PASSWORD") or None

if USERNAME:
    print(f"Using env var credentials for: {USERNAME}")
else:
    creds_file = Path.home() / ".copernicusmarine" / ".copernicusmarine-credentials"
    if creds_file.exists():
        print(f"No env var credentials — will use cached credentials at {creds_file}")
    else:
        print("⚠ No CMEMS credentials found.")
        print("  Run this in a terminal to log in and cache credentials:")
        print("    conda run -n boatops copernicusmarine login")
        print("  Or register at https://marine.copernicus.eu and set COPERNICUSMARINE_SERVICE_USERNAME/PASSWORD in .env")

# AOI — German Baltic coast, centered on Warnemünde
AOI = dict(lon_min=9.5, lat_min=53.5, lon_max=15.0, lat_max=56.0)
LOC = dict(lat=54.18, lon=12.08, name="Warnemünde")

print(f"\ncopernicusmarine version: {cm.__version__}")
print(f"AOI: {AOI}")

## 1. Discover — Find the Right Dataset ID

The product `BALTICSEA_ANALYSISFORECAST_WAV_003_010` may contain multiple datasets
(different temporal resolutions). We list them to pick the right one.

In [ ]:
PRODUCT_ID = "BALTICSEA_ANALYSISFORECAST_WAV_003_010"
DATASET_ID = "cmems_mod_bal_wav_anfc_PT1H-i"  # confirmed: hourly analysis + forecast

# describe() reads credentials from COPERNICUSMARINE_SERVICE_USERNAME/PASSWORD env vars
catalogue = cm.describe(product_id=PRODUCT_ID)

print(f"Product: {PRODUCT_ID}")
for product in catalogue.products:
    print(f"  Title: {product.title}")
    for dataset in product.datasets:
        print(f"  Dataset ID: {dataset.dataset_id}")

In [ ]:
# List variables available in the dataset
dataset_info = cm.describe(dataset_id=DATASET_ID)
for product in dataset_info.products:
    for dataset in product.datasets:
        for version in dataset.versions:
            for part in version.parts:
                for svc in part.services:
                    print(f"Service: {svc.service_name}")
                    for var in (svc.variables or []):
                        print(f"  {var.short_name:12s}  {var.standard_name}")

## 2. Fetch Wave Data — open_dataset (in-memory, no file write)

Use `copernicusmarine.open_dataset()` to load directly into xarray for interactive exploration.
Subset to the Baltic AOI and a 72-hour window (24 h analysis + 48 h forecast).

In [ ]:
# Time window: yesterday → 2 days from now
now_utc  = datetime.now(timezone.utc).replace(minute=0, second=0, microsecond=0)
start_dt = now_utc - timedelta(hours=24)
end_dt   = now_utc + timedelta(hours=48)

print(f"Fetching: {start_dt.strftime('%Y-%m-%d %H:%M')} → {end_dt.strftime('%Y-%m-%d %H:%M')} UTC")
print(f"Variables: VHM0 (Hs), VMDR (mean wave direction)")
print(f"AOI: lon [{AOI['lon_min']}, {AOI['lon_max']}], lat [{AOI['lat_min']}, {AOI['lat_max']}]")

# Pass credentials only if explicitly set; otherwise copernicusmarine uses cached credentials file
auth = dict(username=USERNAME, password=PASSWORD) if USERNAME else {}

ds = cm.open_dataset(
    dataset_id=DATASET_ID,
    variables=["VHM0", "VMDR"],
    minimum_longitude=AOI["lon_min"],
    maximum_longitude=AOI["lon_max"],
    minimum_latitude=AOI["lat_min"],
    maximum_latitude=AOI["lat_max"],
    start_datetime=start_dt,
    end_datetime=end_dt,
    **auth,
)

print(f"\nDataset loaded:")
print(ds)

## 3. Inspect — Shape, Ranges, Metadata (REQ-WAVE-001 through REQ-WAVE-003)

In [ ]:
print("=== Dataset structure ===")
print(f"Dimensions: {dict(ds.dims)}")
print(f"Variables:  {list(ds.data_vars)}")
print(f"Time steps: {len(ds.time)} ({ds.time.values[0]} → {ds.time.values[-1]})")
print(f"Lat range:  {float(ds.latitude.min()):.2f} – {float(ds.latitude.max()):.2f} °N")
print(f"Lon range:  {float(ds.longitude.min()):.2f} – {float(ds.longitude.max()):.2f} °E")
print()

# VHM0 — significant wave height
hs = ds["VHM0"]
print(f"VHM0 (Hs):")
print(f"  Units:    {hs.attrs.get('units', 'unknown')}")
print(f"  Long name: {hs.attrs.get('long_name', 'unknown')}")
print(f"  Range:    {float(hs.min()):.2f} – {float(hs.max()):.2f} m")
print()

# VMDR — mean wave direction
wd = ds["VMDR"]
print(f"VMDR (wave direction):")
print(f"  Units:    {wd.attrs.get('units', 'unknown')}")
print(f"  Long name: {wd.attrs.get('long_name', 'unknown')}")
print(f"  Range:    {float(wd.min()):.1f} – {float(wd.max()):.1f} °")
print()

# REQ-WAVE-003: model run time / last-updated
print(f"Global attributes:")
for k, v in ds.attrs.items():
    print(f"  {k}: {v}")

In [ ]:
# SCENARIO-WAVE-HEIGHT-001: Hs in [0, 10] m range
hs_vals = ds["VHM0"].values
hs_finite = hs_vals[~np.isnan(hs_vals)]
assert hs_finite.min() >= 0,  f"Hs min {hs_finite.min():.2f} m is negative"
assert hs_finite.max() <= 10, f"Hs max {hs_finite.max():.2f} m exceeds 10 m (implausible for Baltic)"
print(f"✓ SCENARIO-WAVE-HEIGHT-001: Hs in [0, 10] m — range: {hs_finite.min():.2f} – {hs_finite.max():.2f} m")

# SCENARIO-WAVE-DIR-001: direction in [0, 360)
wd_vals = ds["VMDR"].values
wd_finite = wd_vals[~np.isnan(wd_vals)]
assert wd_finite.min() >= 0,   f"Direction min {wd_finite.min():.1f} is negative"
assert wd_finite.max() <= 360, f"Direction max {wd_finite.max():.1f} exceeds 360"
print(f"✓ SCENARIO-WAVE-DIR-001: direction in [0, 360] — range: {wd_finite.min():.0f} – {wd_finite.max():.0f} °")

# Time steps: at least 24h of data
assert len(ds.time) >= 24, f"Expected ≥24 time steps, got {len(ds.time)}"
print(f"✓ Time steps: {len(ds.time)} (hourly over {len(ds.time)/24:.1f} days)")

## 4. Time Series at Warnemünde Point

In [ ]:
# Extract nearest grid point to Warnemünde
hs_pt = ds["VHM0"].sel(latitude=LOC["lat"], longitude=LOC["lon"], method="nearest")
wd_pt = ds["VMDR"].sel(latitude=LOC["lat"], longitude=LOC["lon"], method="nearest")

actual_lat = float(hs_pt.latitude)
actual_lon = float(hs_pt.longitude)
print(f"Nearest grid point to {LOC['name']}: ({actual_lat:.2f}°N, {actual_lon:.2f}°E)")

hs_vals_pt = hs_pt.values
nan_frac = np.isnan(hs_vals_pt).mean()
print(f"NaN fraction at nearest point: {nan_frac:.0%}")

if nan_frac > 0.5:
    # Nearest grid point is masked (land/coast) — search for closest wet cell
    print("  ↳ Coastal/land mask — searching for nearest non-NaN point...")
    hs_mean = ds["VHM0"].mean(dim="time")
    wet_mask = ~np.isnan(hs_mean.values)
    lats2d, lons2d = np.meshgrid(ds.latitude.values, ds.longitude.values, indexing="ij")
    dist = (lats2d - LOC["lat"])**2 + (lons2d - LOC["lon"])**2
    dist[~wet_mask] = np.inf
    flat_idx = np.argmin(dist)
    best_lat = ds.latitude.values[flat_idx // len(ds.longitude)]
    best_lon = ds.longitude.values[flat_idx  % len(ds.longitude)]
    hs_pt = ds["VHM0"].sel(latitude=best_lat, longitude=best_lon, method="nearest")
    wd_pt = ds["VMDR"].sel(latitude=best_lat, longitude=best_lon, method="nearest")
    actual_lat, actual_lon = float(hs_pt.latitude), float(hs_pt.longitude)
    print(f"  ↳ Using nearest wet cell: ({actual_lat:.2f}°N, {actual_lon:.2f}°E)")
    hs_vals_pt = hs_pt.values

times     = pd.to_datetime(ds.time.values)
hs_series = hs_vals_pt
wd_series = wd_pt.values
now_naive = pd.Timestamp(now_utc).tz_localize(None)  # tz-naive UTC for time arithmetic

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 6), sharex=True)
fig.suptitle(f"Wave Forecast — nearest wet cell to {LOC['name']} ({actual_lat:.2f}°N, {actual_lon:.2f}°E)", fontsize=12)

ax1.plot(times, hs_series, color="steelblue", lw=2)
ax1.axvline(now_naive, color="gray", ls="--", lw=1, label="Now")
ax1.axhline(0.4, color="orange", ls=":", lw=1, label="0.4 m Go threshold")
ax1.axhline(0.8, color="red",    ls=":", lw=1, label="0.8 m No-Go threshold")
ax1.set_ylabel("Hs (m)")
ax1.set_ylim(bottom=0)
ax1.legend(fontsize=8)
ax1.grid(True, alpha=0.3)

ax2.scatter(times, wd_series, s=8, color="teal", alpha=0.7)
ax2.axvline(now_naive, color="gray", ls="--", lw=1)
ax2.set_ylabel("Wave direction (°)")
ax2.set_ylim(0, 360)
ax2.set_yticks([0, 90, 180, 270, 360])
ax2.set_yticklabels(["N", "E", "S", "W", "N"])
ax2.grid(True, alpha=0.3)
ax2.xaxis.set_major_formatter(mdates.DateFormatter("%a %d %b\n%H:%M"))
ax2.xaxis.set_major_locator(mdates.HourLocator(byhour=[0, 6, 12, 18]))
plt.tight_layout()
plt.show()

current_hs = hs_series[np.argmin(np.abs(times - now_naive))]
print(f"\nCurrent Hs (nearest wet cell to {LOC['name']}): {current_hs:.2f} m")
print(f"Peak forecast Hs: {np.nanmax(hs_series):.2f} m")

## 5. Spatial Map — Hs Snapshot + Wave Direction Arrows (SCENARIO-WAVE-SNAPSHOT-001)

In [ ]:
# SCENARIO-WAVE-SNAPSHOT-001: single time-slice extraction
# ds.time contains tz-naive numpy datetime64; strip tz from now_utc for comparison
now_naive = pd.Timestamp(now_utc).tz_localize(None)
t_now_idx = int(np.argmin(np.abs(pd.to_datetime(ds.time.values) - now_naive)))
snap_time = pd.to_datetime(ds.time.values[t_now_idx])
print(f"Snapshot time: {snap_time} UTC (index {t_now_idx})")

hs_snap = ds["VHM0"].isel(time=t_now_idx)
wd_snap = ds["VMDR"].isel(time=t_now_idx)
lons    = ds.longitude.values
lats    = ds.latitude.values

assert hs_snap.dims == ("latitude", "longitude"), f"Unexpected dims: {hs_snap.dims}"
print(f"✓ SCENARIO-WAVE-SNAPSHOT-001: single time slice shape {hs_snap.shape}")

# Build quiver inputs — subsample for readability
step = max(1, len(lats) // 15)
lon2d, lat2d = np.meshgrid(lons[::step], lats[::step])
wd_sub  = wd_snap.values[::step, ::step]
u_waves = np.sin(np.deg2rad(wd_sub))
v_waves = np.cos(np.deg2rad(wd_sub))

fig, ax = plt.subplots(figsize=(11, 8),
                        subplot_kw=dict(projection=ccrs.PlateCarree()))
ax.set_extent([AOI["lon_min"], AOI["lon_max"],
               AOI["lat_min"], AOI["lat_max"]], crs=ccrs.PlateCarree())

pcm = ax.pcolormesh(lons, lats, hs_snap.values,
                    cmap="Blues", vmin=0, vmax=min(2.0, float(np.nanmax(hs_snap.values)) + 0.1),
                    transform=ccrs.PlateCarree(), shading="auto")
plt.colorbar(pcm, ax=ax, label="Hs (m)", shrink=0.7)

ax.quiver(lon2d, lat2d, u_waves, v_waves,
          transform=ccrs.PlateCarree(),
          scale=25, width=0.003, color="white", alpha=0.8)

ax.add_feature(cfeature.COASTLINE, linewidth=0.8)
ax.add_feature(cfeature.BORDERS,   linewidth=0.4, linestyle=":")
ax.add_feature(cfeature.LAND,      facecolor="#f5f5f0", zorder=1)

ax.plot(LOC["lon"], LOC["lat"], "r^", ms=10, transform=ccrs.PlateCarree(),
        label=LOC["name"], zorder=5)
ax.legend(loc="upper right")

gl = ax.gridlines(draw_labels=True, linewidth=0.4, alpha=0.5)
gl.top_labels = False
gl.right_labels = False

ax.set_title(f"Significant Wave Height + Direction\n{snap_time.strftime('%Y-%m-%d %H:%M')} UTC",
             fontsize=11)
plt.tight_layout()
plt.show()

## 6. Extract Model Run Time — REQ-WAVE-003

In [ ]:
# REQ-WAVE-003: last-updated / model run time
# CMEMS datasets expose this via global attributes or the time dimension metadata
last_updated = None

# Try global attribute keys
for key in ("history", "bulletin_date", "creation_date", "date_created",
            "forecast_reference_time", "time_coverage_start"):
    val = ds.attrs.get(key)
    if val:
        print(f"  {key}: {val}")
        if last_updated is None:
            last_updated = val

# Also record the first and last time step in the dataset as a proxy
t_first = pd.to_datetime(ds.time.values[0]).strftime("%Y-%m-%dT%H:%M UTC")
t_last  = pd.to_datetime(ds.time.values[-1]).strftime("%Y-%m-%dT%H:%M UTC")
print(f"\nData coverage: {t_first} → {t_last}")
print(f"Last-updated proxy: {last_updated or 'not in global attrs — use time.values[-1] as fallback'}")

## 7. Simulated Failure — SCENARIO-WAVE-FAIL-001

In [ ]:
# SCENARIO-WAVE-FAIL-001: verify that API failures return structured error, not a crash.
def fetch_wave_forecast_safe(dataset_id, **kwargs):
    """Wrapper around cm.open_dataset that returns structured result on failure."""
    try:
        data = cm.open_dataset(dataset_id=dataset_id, **kwargs)
        return {"status": "ok", "data": data}
    except Exception as e:
        return {"status": "error", "layer": "waves", "message": str(e)}

result = fetch_wave_forecast_safe("this_dataset_does_not_exist", **auth)
assert result["status"] == "error", "Expected error status for bad dataset ID"
assert result["layer"] == "waves"
print("✓ SCENARIO-WAVE-FAIL-001 PASS")
print(f"  Error message: {result['message'][:120]}")

## 8. Summary of Findings

In [ ]:
print("=== FINDINGS FOR ops/data-sources.md ===")
print()
print(f"copernicusmarine version: {cm.__version__}")
print(f"Product:    {PRODUCT_ID}")
print(f"Dataset ID: {DATASET_ID}")
print(f"Variables confirmed: VHM0 (Hs, m), VMDR (mean wave direction, °)")
print(f"Spatial resolution: ~{abs(float(ds.latitude[1] - ds.latitude[0])):.3f}° lat × "
      f"{abs(float(ds.longitude[1] - ds.longitude[0])):.3f}° lon")
print(f"Time resolution: {(pd.to_datetime(ds.time.values[1]) - pd.to_datetime(ds.time.values[0])).total_seconds()/3600:.0f} h")
print(f"Coverage in AOI: {len(ds.time)} time steps over {len(ds.time)/24:.1f} days")
print()
print("Auth:")
print("  Uses COPERNICUSMARINE_SERVICE_USERNAME/PASSWORD (same Copernicus account as CDSE)")
print("  Credentials confirmed working ✓")
print()
print("Phase 2 design notes:")
print("  - Use cm.open_dataset() (in-memory) for backend API calls — no temp file needed")
print("  - Pass username/password directly from env vars — no credentials file on server")
print("  - For cache: TTL = 1 h matches product update cadence")
print("  - Model run time: check global attrs; fallback to ds.time.values[-1]")
print("  - Land/coast cells return NaN — handle in visualisation (mask with np.isnan)")